In [0]:
from pyspark.sql.functions import window, avg, min, max, sum, col, lit


silver = spark.table("workspace4sadt.silver.sensors_cleaned") \
    .filter(col("is_valid") == True)

gold = silver.groupBy(
    "district",
    "topic",
    window("sensor_timestamp", "5 minutes")
).agg(
    avg("value_si").alias("avg_value"),
    min("value_si").alias("min_value"),
    max("value_si").alias("max_value"),
    sum(col("alert_flag").cast("int")).alias("alert_count"),
    sum(lit(1)).alias("total_count")
)

gold = gold.select(
    "district",
    "topic",
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    "avg_value",
    "min_value",
    "max_value",
    "alert_count",
    "total_count"
)

gold.write.format("delta") \
    .mode("append") \
    .saveAsTable("workspace4sadt.gold.sensors_agg")


In [0]:
%sql
-- SELECT * FROM workspace4sadt.gold.sensors_agg;

district,topic,window_start,window_end,avg_value,min_value,max_value,alert_count,total_count
3,null,2025-12-19T18:15:00.000Z,2025-12-19T18:20:00.000Z,13.916666666666666,13.916666666666666,13.916666666666666,0,2
3,utilities.sensors,2025-12-19T18:25:00.000Z,2025-12-19T18:30:00.000Z,36.01923793482398,17.632530251112897,58.38412992515744,0,4
4,greeninfra.sensors,2025-12-19T18:25:00.000Z,2025-12-19T18:30:00.000Z,47.74825769817636,10.820643319304335,80.71189576855588,1,10
5,traffic.sensors,2025-12-19T18:25:00.000Z,2025-12-19T18:30:00.000Z,12.985617623541739,4.272504036355461,25.66713084889426,1,8
1,transport.sensors,2025-12-19T18:30:00.000Z,2025-12-19T18:35:00.000Z,54.84694032060386,54.84694032060386,54.84694032060386,0,1
1,traffic.sensors,2025-12-19T18:25:00.000Z,2025-12-19T18:30:00.000Z,15.861804877359384,4.09527665746899,23.014781330982995,1,10
2,environment.sensors,2025-12-19T18:30:00.000Z,2025-12-19T18:35:00.000Z,47.966397180449775,11.08574610179482,87.61586596916062,1,6
3,environment.sensors,2025-12-19T18:25:00.000Z,2025-12-19T18:30:00.000Z,59.073489919195396,16.238751250015834,99.43940537450779,2,4
2,greeninfra.sensors,2025-12-19T18:25:00.000Z,2025-12-19T18:30:00.000Z,72.09294738495524,18.49963992540151,95.73315452301348,5,10
3,greeninfra.sensors,2025-12-19T18:30:00.000Z,2025-12-19T18:35:00.000Z,73.02074363825439,46.48412811864855,99.55735915786023,1,2
